# Entrega 2 — comparación de modelos para enfermedades en hojas de tomate

**Objetivo.** Comparar la línea base de histogramas HSV + regresión logística de la primera entrega con una CNN pequeña y MobileNetV2 mediante transferencia de aprendizaje.

**Protocolo.** `train` ajusta los modelos; `val` guía la selección y la parada temprana; `test` se reserva para una única evaluación final. El aumento de datos se aplica solo al entrenamiento. El conjunto procesado tiene seis clases y deriva de PlantVillage (Shovo y Mostafa, DOI [10.17632/3zwdw6y4pn.1](https://doi.org/10.17632/3zwdw6y4pn.1), CC BY 4.0). No se incluyen métricas precomputadas: los resultados deben generarse al ejecutar el notebook.

**Entorno:** Python 3.10+, TensorFlow, scikit-learn, OpenCV, Matplotlib y Seaborn. Fija y registra la semilla, versiones, fecha y hardware de la ejecución antes de entregar resultados.

In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# Busca las rutas documentadas del repositorio y de la primera entrega en Colab.
candidates = [Path("data/raw/Dataset"), Path("dataset_tomato/Dataset"), Path("data/raw")]
base_dir = next((p for p in candidates if all((p / s).is_dir() for s in ("train", "val", "test"))), None)
if base_dir is None:
    raise FileNotFoundError("No se encontró Dataset/train, val y test. Descarga con: python -m src.data --download --destination data/raw")
print("Dataset:", base_dir.resolve())


## 1. Revisión de particiones y clases

Comprobamos que las tres particiones contienen las mismas clases antes de entrenar.

In [ ]:
SPLITS = ("train", "val", "test")
counts = []
classes_by_split = {}
for split in SPLITS:
    split_path = base_dir / split
    classes_by_split[split] = sorted(p.name for p in split_path.iterdir() if p.is_dir())
    for class_name in classes_by_split[split]:
        n_images = sum(1 for p in (split_path / class_name).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
        counts.append({"Partición": split, "Clase": class_name, "Imágenes": n_images})
if not classes_by_split["train"] or any(classes_by_split[s] != classes_by_split["train"] for s in SPLITS[1:]):
    raise ValueError(f"Clases inconsistentes entre particiones: {classes_by_split}")
class_names = classes_by_split["train"]
display(pd.DataFrame(counts).pivot(index="Clase", columns="Partición", values="Imágenes").fillna(0).astype(int))


## 2. Generadores y aumento de datos

La CNN propia recibe píxeles en `[0, 1]`. MobileNetV2 usa su preprocesamiento oficial, que transforma imágenes al rango esperado por los pesos ImageNet. Validación y prueba no reciben aumento.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 25

train_cnn = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=.12,
    height_shift_range=.12, zoom_range=.12, horizontal_flip=True, fill_mode="nearest")
plain_cnn = ImageDataGenerator(rescale=1./255)
train_tl = ImageDataGenerator(preprocessing_function=preprocess_input, rotation_range=20,
    width_shift_range=.12, height_shift_range=.12, zoom_range=.12, horizontal_flip=True, fill_mode="nearest")
plain_tl = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_generator(datagen, split, shuffle):
    return datagen.flow_from_directory(str(base_dir / split), target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode="categorical", classes=class_names,
        shuffle=shuffle, seed=SEED)

train_cnn_gen = make_generator(train_cnn, "train", True)
val_cnn_gen = make_generator(plain_cnn, "val", False)
test_cnn_gen = make_generator(plain_cnn, "test", False)
train_tl_gen = make_generator(train_tl, "train", True)
val_tl_gen = make_generator(plain_tl, "val", False)
test_tl_gen = make_generator(plain_tl, "test", False)


## 3. Línea base HSV + regresión logística

Reutilizamos el extractor de la primera entrega. El modelo se ajusta solo con `train`; reportamos validación para contextualizar selección. La evaluación comparativa definitiva ocurre junto a las redes, al final.

In [ ]:
from src.features import load_split_features
from sklearn.linear_model import LogisticRegression

X_train, y_train, baseline_classes = load_split_features(base_dir, "train")
X_val, y_val, _ = load_split_features(base_dir, "val")
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
print(classification_report(y_val, baseline.predict(X_val), labels=baseline_classes, zero_division=0))


## 4. CNN pequeña

Entrenamos una CNN propia con parada temprana según pérdida de validación.

In [ ]:
def build_cnn(num_classes):
    return models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(), layers.Dense(128, activation="relu"),
        layers.Dropout(.4), layers.Dense(len(class_names), activation="softmax")
    ])

cnn = build_cnn(len(class_names))
cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
history_cnn = cnn.fit(train_cnn_gen, validation_data=val_cnn_gen, epochs=EPOCHS,
    callbacks=[early_stop], verbose=1)


## 5. Transfer learning con MobileNetV2

Se congela la base ImageNet y se entrena una cabeza clasificadora. El tamaño 128×128 reduce el costo; `preprocess_input` es específico de MobileNetV2.

In [ ]:
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False
inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
transfer_model = models.Model(inputs, outputs)
transfer_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history_tl = transfer_model.fit(train_tl_gen, validation_data=val_tl_gen, epochs=EPOCHS,
    callbacks=[early_stop], verbose=1)


## 6. Evaluación final en test

Esta celda realiza la evaluación final después del entrenamiento y selección. Se alinea el baseline HSV y las redes sobre las mismas imágenes `test`. No uses estos resultados para seguir ajustando hiperparámetros.

In [ ]:
X_test, y_test, _ = load_split_features(base_dir, "test")
predictions = {
    "Baseline HSV + Regresión logística": baseline.predict(X_test),
}
test_cnn_gen.reset()
predictions["CNN pequeña + augmentation"] = np.asarray(class_names)[np.argmax(cnn.predict(test_cnn_gen, verbose=1), axis=1)]
test_tl_gen.reset()
predictions["MobileNetV2 + transfer learning"] = np.asarray(class_names)[np.argmax(transfer_model.predict(test_tl_gen, verbose=1), axis=1)]

rows = []
for name, y_pred in predictions.items():
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=class_names, average="macro", zero_division=0)
    rows.append({"Modelo": name, "Accuracy": accuracy_score(y_test, y_pred),
                 "Precision macro": precision, "Recall macro": recall, "F1 macro": f1})
benchmark = pd.DataFrame(rows).sort_values("F1 macro", ascending=False)
display(benchmark.round(4))


In [ ]:
for name, y_pred in predictions.items():
    print(f"\n### {name}")
    print(classification_report(y_test, y_pred, labels=class_names, target_names=class_names, digits=4, zero_division=0))
    cm = confusion_matrix(y_test, y_pred, labels=class_names)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Matriz de confusión — {name}")
    plt.xlabel("Predicción"); plt.ylabel("Etiqueta real")
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()


## 7. Resultados comunicados en el segundo corte

Los siguientes valores se transcribieron de la presentación y la libreta original que están en `../2° avance/`; no son salidas de esta ejecución. En el material fuente, baseline HSV se midió en validación (n=981), mientras CNN y MobileNetV2 se midieron en test (n=990). No interpretes el orden de esos tres modelos como una comparación equitativa.

| Modelo | Partición fuente | n | Accuracy | Precision macro | Recall macro | F1 macro |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| HSV + regresión logística | Validation | 981 | 0.906218 | 0.905382 | 0.906074 | 0.904660 |
| CNN pequeña + augmentation | Test | 990 | 0.943434 | 0.946009 | 0.943316 | 0.942456 |
| MobileNetV2 | Test | 990 | 0.884848 | 0.887306 | 0.884613 | 0.883918 |

Entre CNN y MobileNetV2, evaluadas ambas en test, la CNN queda 5.86 puntos porcentuales por encima en accuracy y 0.0585 en macro F1. La clase más débil para la CNN es Leaf Mold (recall 0.80; 131/164 aciertos, 19 confundidos con Septoria). Para MobileNetV2 es Early blight (recall 0.73; 119/164 aciertos). El reporte ampliado está en `reports/second_cut_results.md`.

MobileNetV2 del experimento original recibió imágenes reescaladas a `[0, 1]`; sus pesos ImageNet esperan `preprocess_input`. Este notebook corrige ese preprocesamiento, por lo que sus resultados reejecutados pueden cambiar. La tabla anterior conserva el registro histórico y debe reemplazarse con una comparación reproducida en test para los tres modelos.

## 8. Registro e interpretación

Registra los resultados generados, fecha, versiones de bibliotecas, hardware, duración de entrenamiento, semilla, pesos utilizados y cualquier desviación del protocolo en `reports/`. Interpreta el macro F1 junto con los errores por clase y las matrices de confusión; una puntuación global no describe todos los riesgos.

**Limitaciones:** las imágenes de laboratorio pueden no representar condiciones de campo; el aumento sintético no resuelve el cambio de dominio; los pesos preentrenados requieren descarga; y el diagnóstico automático es apoyo académico, no reemplaza evaluación agronómica. Conserva la atribución CC BY 4.0 del dataset.